In [1]:
# ==========================================================
# BASE DE DATOS SUCIA - SIMULACIÓN PRUEBA TÉCNICA FLUID ATTACKS
# Incluye: NULLs, Nulos Implícitos ('N/A', 'UNKNOWN', 'NaN'),
# Duplicados, Inconsistencias de Tipo y Fechas Mal Formateadas.
# ==========================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .appName("FluidAttacks_RealData_Prep") \
    .master("local[*]") \
    .getOrCreate()

# ----------------------------------------------------------
# TABLA 1: proyectos (Con nulos y datos faltantes)
# ----------------------------------------------------------
proyectos_data = [
    (101, "Fintech-API", "Backend", "Alta"),
    (102, "ECommerce-Web", "Frontend", None),        # NULL en criticidad
    (103, "Auth-Service", None, "Crítica"),           # NULL en tipo
    (104, "Data-Pipeline", "Data", "UNKNOWN"),        # Nulo implícito
    (105, "Legacy-Portal", "Web", "Crítica")
]
proyectos_schema = ["id_proyecto", "nombre_proyecto", "tipo", "criticidad_negocio"]

# ----------------------------------------------------------
# TABLA 2: hallazgos (Data Semiautomática de Scanners de Seguridad)
# ----------------------------------------------------------
hallazgos_schema = StructType([
    StructField("id_hallazgo", IntegerType(), True),
    StructField("id_proyecto", IntegerType(), True),
    StructField("vulnerabilidad", StringType(), True),
    StructField("severidad", StringType(), True),
    StructField("cvss_score", DoubleType(), True),    # Nulos explícitos
    StructField("fecha_deteccion", StringType(), True),# Formatos de fecha mixtos
    StructField("fecha_remediacion", StringType(), True),
    StructField("estado", StringType(), True)
])

hallazgos_data = [
    # Normales y resueltos
    (1, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Sin fecha de remediación ni CVSS (Abierta)
    (2, 101, "XSS Cross-Site Scripting", "HIGH", None, "2026-01-15", None, "OPEN"),
    # Nulo implícito como String ('NULL' o 'N/A')
    (3, 101, "Hardcoded Credentials", "CRITICAL", 8.9, "2026-02-01", "N/A", "REMEDIATED"),
    # Formato de fecha distinto (MM/DD/YYYY)
    (4, 102, "Insecure Dependency", "MEDIUM", 5.3, "01/20/2026", "02/15/2026", "REMEDIATED"),
    # CVSS Nulo
    (5, 102, "XSS Cross-Site Scripting", "MEDIUM", None, "2026-02-10", None, "OPEN"),
    # SLA ultra rápido (0 días)
    (6, 103, "Remote Code Execution", "CRITICAL", 10.0, "2026-01-05", "2026-01-05", "REMEDIATED"),
    # Sin severidad reportada
    (7, 103, "Broken Access Control", None, 8.1, "2026-01-12", None, "IN_PROGRESS"),
    # Duplicado exacto del ID 1 (Filtro requerido)
    (1, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Duplicado lógico (Diferente ID, mismo evento registrado 2 veces)
    (13, 101, "SQL Injection", "CRITICAL", 9.8, "2026-01-10", "2026-01-12", "REMEDIATED"),
    # Outlier en fecha (Fecha futura o errónea de logs)
    (8, 104, "CSRF", "LOW", 3.2, "2030-01-01", None, "OPEN"),
    # Sin proyecto asignado (Orfano - probar RIGHT/FULL JOINs)
    (9, 999, "Unauthenticated Access", "HIGH", 8.7, "2026-02-01", None, "OPEN")
]

# ----------------------------------------------------------
# TABLA 3: logs_autenticacion (Inyecciones de Brute Force)
# ----------------------------------------------------------
logs_data = [
    (1001, "dev_user1", "192.168.1.10", "SUCCESS", "2026-02-20 08:30:00"),
    (1002, "admin_sec", None, "SUCCESS", "2026-02-20 08:32:00"),             # IP Nula
    (1003, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:00"),
    (1004, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:10"),
    (1005, "attacker_x", "45.33.22.11", "FAILED", "2026-02-20 08:35:20"),
    (1006, "attacker_x", "45.33.22.11", "SUCCESS", "2026-02-20 08:36:00"),  # Ataque exitoso
    (1007, None, "185.220.101.5", "FAILED", "2026-02-20 09:15:00")            # Usuario Nulo
]
logs_schema = ["id_log", "usuario", "ip_origen", "resultado", "timestamp_acceso"]

# Crear DataFrames
df_proyectos = spark.createDataFrame(proyectos_data, proyectos_schema)
df_hallazgos = spark.createDataFrame(hallazgos_data, hallazgos_schema)
df_logs = spark.createDataFrame(logs_data, logs_schema)

# Crear Vistas SQL
df_proyectos.createOrReplaceTempView("proyectos")
df_hallazgos.createOrReplaceTempView("hallazgos")
df_logs.createOrReplaceTempView("logs_autenticacion")

print("¡Base de Datos REALISTA cargada!")    

¡Base de Datos REALISTA cargada!


In [2]:
query = """
SELECT *
FROM proyectos
"""
spark.sql(query).show()

+-----------+---------------+--------+------------------+
|id_proyecto|nombre_proyecto|    tipo|criticidad_negocio|
+-----------+---------------+--------+------------------+
|        101|    Fintech-API| Backend|              Alta|
|        102|  ECommerce-Web|Frontend|              NULL|
|        103|   Auth-Service|    NULL|           Crítica|
|        104|  Data-Pipeline|    Data|           UNKNOWN|
|        105|  Legacy-Portal|     Web|           Crítica|
+-----------+---------------+--------+------------------+



In [3]:
query = """
SELECT *
FROM hallazgos
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12| REMEDIATED|
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A| REMEDIATED|
|          4|        102| Insecure Dependency|   MEDIUM|       5.3|     01/20/2026|       02/15/2026| REMEDIATED|
|          5|        102|XSS Cross-Site Sc...|   MEDIUM|      NULL|     2026-02-10|             NULL|       OPEN|
|          6|        103|Remote Code Execu...| CRITICAL|      10.0|     2026-01-05|     

In [4]:
query = """
SELECT *
FROM logs_autenticacion
"""
spark.sql(query).show()

+------+----------+-------------+---------+-------------------+
|id_log|   usuario|    ip_origen|resultado|   timestamp_acceso|
+------+----------+-------------+---------+-------------------+
|  1001| dev_user1| 192.168.1.10|  SUCCESS|2026-02-20 08:30:00|
|  1002| admin_sec|         NULL|  SUCCESS|2026-02-20 08:32:00|
|  1003|attacker_x|  45.33.22.11|   FAILED|2026-02-20 08:35:00|
|  1004|attacker_x|  45.33.22.11|   FAILED|2026-02-20 08:35:10|
|  1005|attacker_x|  45.33.22.11|   FAILED|2026-02-20 08:35:20|
|  1006|attacker_x|  45.33.22.11|  SUCCESS|2026-02-20 08:36:00|
|  1007|      NULL|185.220.101.5|   FAILED|2026-02-20 09:15:00|
+------+----------+-------------+---------+-------------------+



In [5]:
"""
Ejercicio 1 — Filtro simple con múltiples condiciones
Lista los hallazgos con severidad = 'HIGH' y estado = 'OPEN'. Conceptos: WHERE, operador AND, comparación de strings. Pista de verificación: 
deberías obtener 2 filas. Una de ellas pertenece a un proyecto que no existe en la tabla proyectos — no hace falta que lo notes todavía, 
solo recuérdalo para más adelante.
"""
query = """
SELECT
    *
FROM hallazgos AS ha
LEFT JOIN proyectos AS py
    ON ha.id_proyecto = py.id_proyecto
WHERE
    ha.severidad = 'HIGH'
    AND ha.estado = 'OPEN'
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+------+-----------+---------------+-------+------------------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|estado|id_proyecto|nombre_proyecto|   tipo|criticidad_negocio|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+------+-----------+---------------+-------+------------------+
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|  OPEN|        101|    Fintech-API|Backend|              Alta|
|          9|        999|Unauthenticated A...|     HIGH|       8.7|     2026-02-01|             NULL|  OPEN|       NULL|           NULL|   NULL|              NULL|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+------+-----------+---------------+-------+------------------+



In [6]:
"""Ejercicio 2 — Ordenar y limitar resultados
Objetivo: Muestra los 5 hallazgos con mayor cvss_score, de mayor a menor. Decide qué hacer con los valores nulos de cvss_score en el orden 
(¿deberían aparecer primero, al final, o excluirse?) y justifica tu elección. Conceptos: ORDER BY, LIMIT, comportamiento de NULL en ordenamientos 
(NULLS FIRST/LAST)."""

query = """
SELECT
    COUNT(CASE 
            WHEN ha.cvss_score IS NULL THEN 1 ELSE NULL 
          END)
FROM hallazgos AS ha
"""
spark.sql(query).show()

+----------------------------------------------------------+
|count(CASE WHEN (cvss_score IS NULL) THEN 1 ELSE NULL END)|
+----------------------------------------------------------+
|                                                         2|
+----------------------------------------------------------+



In [7]:
query = """
SELECT
    *
FROM hallazgos AS ha
WHERE
    ha.cvss_score IS NOT NULL
ORDER BY
    ha.cvss_score DESC
LIMIT 5
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|    estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+----------+
|          6|        103|Remote Code Execu...| CRITICAL|      10.0|     2026-01-05|       2026-01-05|REMEDIATED|
|         13|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A|REMEDIATED|
+-----------+-----------+--------------------+---------+----------+---------------+-------------

In [8]:
"""
Ejercicio 3 — Nulos explícitos
Objetivo: Encuentra todos los hallazgos que siguen sin fecha de remediación (fecha_remediacion IS NULL) y que además siguen abiertos o en progreso. 
Conceptos: IS NULL, IN o múltiples OR. Pista: ten cuidado — un hallazgo puede tener fecha_remediacion = 'N/A' (string, no NULL real). 
Ese no debería aparecer en este ejercicio si filtras solo por IS NULL. Anota esa diferencia, la vas a necesitar en el Ejercicio 8.
"""

query = """
SELECT
    *
FROM hallazgos AS ha
WHERE ha.fecha_remediacion IS NULL
    AND ha.estado IN ('OPEN', 'IN_PROGRESS')
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|
|          5|        102|XSS Cross-Site Sc...|   MEDIUM|      NULL|     2026-02-10|             NULL|       OPEN|
|          7|        103|Broken Access Con...|     NULL|       8.1|     2026-01-12|             NULL|IN_PROGRESS|
|          8|        104|                CSRF|      LOW|       3.2|     2030-01-01|             NULL|       OPEN|
|          9|        999|Unauthenticated A...|     HIGH|       8.7|     2026-02-01|             NULL|       OPEN|
+-----------+-----------+--------------------+---------+----------+---------------+-----

In [9]:
"""
Ejercicio 4 — Sustituir nulos con COALESCE
Objetivo: Muestra id_hallazgo, vulnerabilidad y severidad, pero donde severidad sea NULL, muestra el texto 'SIN CLASIFICAR' en su lugar. Conceptos: COALESCE.
"""

query = """
SELECT
    ha.id_hallazgo AS ID,
    ha.vulnerabilidad AS vulnerabilidad,
    COALESCE(ha.severidad, 'SIN CLASIFICAR') AS severidad
FROM hallazgos AS ha
"""
spark.sql(query).show()

+---+--------------------+--------------+
| ID|      vulnerabilidad|     severidad|
+---+--------------------+--------------+
|  1|       SQL Injection|      CRITICAL|
|  2|XSS Cross-Site Sc...|          HIGH|
|  3|Hardcoded Credent...|      CRITICAL|
|  4| Insecure Dependency|        MEDIUM|
|  5|XSS Cross-Site Sc...|        MEDIUM|
|  6|Remote Code Execu...|      CRITICAL|
|  7|Broken Access Con...|SIN CLASIFICAR|
|  1|       SQL Injection|      CRITICAL|
| 13|       SQL Injection|      CRITICAL|
|  8|                CSRF|           LOW|
|  9|Unauthenticated A...|          HIGH|
+---+--------------------+--------------+



In [10]:
"""
Ejercicio 5 — Agregación simple
Objetivo: Cuenta cuántos hallazgos hay por cada valor de estado. Conceptos: COUNT(*), GROUP BY. Pista: el conteo total de filas de la tabla incluye un 
duplicado exacto (mismo id_hallazgo). Este ejercicio no pide que lo filtres — solo obsérvalo, porque en el Ejercicio 14 vas a tener que limpiarlo.
"""

query = """
SELECT 
    ha.estado AS estado,
    COUNT(ha.estado)
FROM hallazgos AS ha
GROUP BY
    ha.estado
"""
spark.sql(query).show()

+-----------+-------------+
|     estado|count(estado)|
+-----------+-------------+
| REMEDIATED|            6|
|       OPEN|            4|
|IN_PROGRESS|            1|
+-----------+-------------+



In [11]:
"""
Ejercicio 6 — Múltiples funciones de agregación
Objetivo: Por cada id_proyecto, calcula el promedio, el máximo y el mínimo de cvss_score, redondeando el promedio a 2 decimales. 
Conceptos: AVG, MAX, MIN, ROUND, GROUP BY. Pista: las funciones de agregación ignoran los NULL automáticamente — confirma que entiendes por qué el 
promedio no se ve afectado por las filas con cvss_score = NULL.
"""

query = """
SELECT
    COUNT(CASE WHEN ha.cvss_score IS NULL THEN 1 ELSE NULL END)
FROM hallazgos AS ha
"""
spark.sql(query).show()

+----------------------------------------------------------+
|count(CASE WHEN (cvss_score IS NULL) THEN 1 ELSE NULL END)|
+----------------------------------------------------------+
|                                                         2|
+----------------------------------------------------------+



In [12]:
query = """
SELECT
    ha.id_proyecto AS proyecto,
    ROUND(AVG(COALESCE(ha.cvss_score, 0)), 2) AS AVG,
    MAX(ha.cvss_score) AS MAX,
    MIN(ha.cvss_score) AS MIN
FROM hallazgos AS ha
GROUP BY
    ha.id_proyecto
"""
spark.sql(query).show()

+--------+----+----+---+
|proyecto| AVG| MAX|MIN|
+--------+----+----+---+
|     101|7.66| 9.8|8.9|
|     102|2.65| 5.3|5.3|
|     103|9.05|10.0|8.1|
|     104| 3.2| 3.2|3.2|
|     999| 8.7| 8.7|8.7|
+--------+----+----+---+



In [13]:
"""
Ejercicio 7 — Filtrar grupos con HAVING
Objetivo: Encuentra los proyectos que tienen más de 2 hallazgos registrados. Conceptos: GROUP BY + HAVING, diferencia entre filtrar filas (WHERE) y 
filtrar grupos (HAVING). Pista: piensa en qué orden se evalúan WHERE y HAVING según el orden de ejecución del motor que ya estudiaste — eso determina si 
el duplicado exacto del Ejercicio 5 infla o no este conteo.
"""

query = """
SELECT
    ha.id_proyecto AS proyecto,
    COUNT(ha.id_hallazgo) AS num_hallazgo
FROM hallazgos AS ha
GROUP BY
    ha.id_proyecto
HAVING
    COUNT(ha.id_hallazgo) > 2
"""
spark.sql(query).show()

+--------+------------+
|proyecto|num_hallazgo|
+--------+------------+
|     101|           5|
+--------+------------+



In [14]:
# sin registros duplicados
query = """
SELECT
    ha.id_proyecto AS proyecto,
    COUNT(DISTINCT ha.id_hallazgo) AS num_hallazgo
FROM hallazgos AS ha
GROUP BY
    ha.id_proyecto
HAVING
    COUNT(ha.id_hallazgo) > 2
"""
spark.sql(query).show()

+--------+------------+
|proyecto|num_hallazgo|
+--------+------------+
|     101|           4|
+--------+------------+



In [15]:
"""Ejercicio 8 — Cazar nulos implícitos
Objetivo: Encuentra todas las filas de proyectos y hallazgos donde algún campo de texto tenga un "nulo disfrazado": 'N/A', 'UNKNOWN', 'NaN' 
(revisa criticidad_negocio, tipo, fecha_remediacion, severidad). Conceptos: IN, OR, UPPER() para normalizar mayúsculas/minúsculas antes de comparar. 
Pista: un WHERE columna = 'N/A' bien escrito no va a detectar estos casos si no cubres las variantes de mayúsculas o si olvidas alguna columna. 
Este es el ejercicio que más se parece a un problema real de calidad de datos."""

query = """
SELECT
    *
FROM proyectos AS py
WHERE py.criticidad_negocio IS NULL
    OR UPPER(py.criticidad_negocio) IN ('UNKNOWN', 'N/A')
"""
spark.sql(query).show()

+-----------+---------------+--------+------------------+
|id_proyecto|nombre_proyecto|    tipo|criticidad_negocio|
+-----------+---------------+--------+------------------+
|        102|  ECommerce-Web|Frontend|              NULL|
|        104|  Data-Pipeline|    Data|           UNKNOWN|
+-----------+---------------+--------+------------------+



In [16]:
query = """
SELECT
    DISTINCT ha.severidad
FROM hallazgos AS ha
"""
spark.sql(query).show()

+---------+
|severidad|
+---------+
| CRITICAL|
|     HIGH|
|   MEDIUM|
|      LOW|
|     NULL|
+---------+



In [17]:
query = """
SELECT
    DISTINCT ha.fecha_remediacion
FROM hallazgos AS ha
"""
spark.sql(query).show()

+-----------------+
|fecha_remediacion|
+-----------------+
|       2026-01-12|
|              N/A|
|       02/15/2026|
|       2026-01-05|
|             NULL|
+-----------------+



In [18]:
query = """
SELECT
    *
FROM hallazgos AS ha
WHERE ha.fecha_remediacion IS NULL
    OR UPPER(ha.fecha_remediacion) IN ('UNKNOWN', 'N/A')
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A| REMEDIATED|
|          5|        102|XSS Cross-Site Sc...|   MEDIUM|      NULL|     2026-02-10|             NULL|       OPEN|
|          7|        103|Broken Access Con...|     NULL|       8.1|     2026-01-12|             NULL|IN_PROGRESS|
|          8|        104|                CSRF|      LOW|       3.2|     2030-01-01|             NULL|       OPEN|
|          9|        999|Unauthenticated A...|     HIGH|       8.7|     2026-02-01|     

In [ ]:
"""
Ejercicio 9 — Clasificación con CASE WHEN
Objetivo: Crea una columna nivel_riesgo que clasifique cada hallazgo según su cvss_score: 
9.0–10.0 = 'Crítico', 7.0–8.9 = 'Alto', 4.0–6.9 = 'Medio', 0.1–3.9 = 'Bajo', y si cvss_score es NULL = 'Sin evaluar'. 
Conceptos: CASE WHEN ... THEN ... END, rangos con BETWEEN o comparadores, manejo del caso NULL dentro del CASE.
"""

In [19]:
query = """
SELECT
    *
FROM hallazgos AS ha
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12| REMEDIATED|
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A| REMEDIATED|
|          4|        102| Insecure Dependency|   MEDIUM|       5.3|     01/20/2026|       02/15/2026| REMEDIATED|
|          5|        102|XSS Cross-Site Sc...|   MEDIUM|      NULL|     2026-02-10|             NULL|       OPEN|
|          6|        103|Remote Code Execu...| CRITICAL|      10.0|     2026-01-05|     

In [21]:
query = """
SELECT
    *,
    (
    CASE 
        WHEN (COALESCE(ha.cvss_score, 0) BETWEEN 0 AND 3.9) THEN 'Bajo'
        WHEN (COALESCE(ha.cvss_score, 0) BETWEEN 4.0 AND 6.9) THEN 'Medio'
        WHEN (COALESCE(ha.cvss_score, 0) BETWEEN 7.0 AND 8.9) THEN 'Alto'
        WHEN (COALESCE(ha.cvss_score, 0) BETWEEN 9.0 AND 10.0) THEN 'Critico'   
    END
    ) AS nivel_riesgo
FROM hallazgos AS ha
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+------------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|     estado|nivel_riesgo|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+-----------+------------+
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12| REMEDIATED|     Critico|
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|       OPEN|        Bajo|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A| REMEDIATED|        Alto|
|          4|        102| Insecure Dependency|   MEDIUM|       5.3|     01/20/2026|       02/15/2026| REMEDIATED|       Medio|
|          5|        102|XSS Cross-Site Sc...|   MEDIUM|      NULL|     2026-02-10|             NULL|       OPE

In [ ]:
"""
Ejercicio 10 — Jerarquía de operadores lógicos
Objetivo: Encuentra los hallazgos que sean (severidad = 'CRITICAL' Y estado = 'OPEN') O (severidad = 'HIGH' Y estado = 'IN_PROGRESS'). 
Conceptos: precedencia de AND sobre OR, uso obligatorio de paréntesis para expresar la intención real. Pista: escribe la consulta primero sin paréntesis 
y compara el resultado con la versión correcta — vas a ver en carne propia por qué la jerarquía de operadores importa tanto como la sintaxis.
"""

In [25]:
query = """
SELECT
    *
FROM hallazgos AS ha
WHERE
    (ha.severidad = 'CRITICAL' AND ha.estado = 'REMEDIATED')
    OR (ha.severidad = 'HIGH' AND ha.estado = 'OPEN')
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|    estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+----------+
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|      OPEN|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A|REMEDIATED|
|          6|        103|Remote Code Execu...| CRITICAL|      10.0|     2026-01-05|       2026-01-05|REMEDIATED|
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|         13|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-0

In [26]:
query = """
SELECT
    *
FROM hallazgos AS ha
WHERE
    ha.severidad = 'CRITICAL' AND ha.estado = 'REMEDIATED'
    OR ha.severidad = 'HIGH' AND ha.estado = 'OPEN'
"""
spark.sql(query).show()

+-----------+-----------+--------------------+---------+----------+---------------+-----------------+----------+
|id_hallazgo|id_proyecto|      vulnerabilidad|severidad|cvss_score|fecha_deteccion|fecha_remediacion|    estado|
+-----------+-----------+--------------------+---------+----------+---------------+-----------------+----------+
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|          2|        101|XSS Cross-Site Sc...|     HIGH|      NULL|     2026-01-15|             NULL|      OPEN|
|          3|        101|Hardcoded Credent...| CRITICAL|       8.9|     2026-02-01|              N/A|REMEDIATED|
|          6|        103|Remote Code Execu...| CRITICAL|      10.0|     2026-01-05|       2026-01-05|REMEDIATED|
|          1|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-01-12|REMEDIATED|
|         13|        101|       SQL Injection| CRITICAL|       9.8|     2026-01-10|       2026-0

In [ ]:
"""Ejercicio 11 — INNER JOIN básico
Objetivo: Combina hallazgos con proyectos para mostrar nombre_proyecto, vulnerabilidad y severidad de cada hallazgo. Conceptos: INNER JOIN ... 
ON. Pista: cuenta las filas del resultado y compáralas con el total de hallazgos. Si el número es distinto, hay una razón — anótala, es la base del 
Ejercicio 13."""

In [27]:
query = """
SELECT
    py.nombre_proyecto AS proyecto,
    ha.vulnerabilidad AS vulnerabilidad,
    ha.severidad AS severidad
FROM hallazgos AS ha
INNER JOIN proyectos AS py
    ON ha.id_proyecto = py.id_proyecto
"""
spark.sql(query).show()

+-------------+--------------------+---------+
|     proyecto|      vulnerabilidad|severidad|
+-------------+--------------------+---------+
|  Fintech-API|       SQL Injection| CRITICAL|
|  Fintech-API|XSS Cross-Site Sc...|     HIGH|
|  Fintech-API|Hardcoded Credent...| CRITICAL|
|  Fintech-API|       SQL Injection| CRITICAL|
|  Fintech-API|       SQL Injection| CRITICAL|
|ECommerce-Web| Insecure Dependency|   MEDIUM|
|ECommerce-Web|XSS Cross-Site Sc...|   MEDIUM|
| Auth-Service|Remote Code Execu...| CRITICAL|
| Auth-Service|Broken Access Con...|     NULL|
|Data-Pipeline|                CSRF|      LOW|
+-------------+--------------------+---------+

